# Stage 5C — Baseline and first model

**Goal:** Predict `1` when a request will still be open after 72 hours. First make a simple baseline, then train a model and compare them on later requests.

**Why:** A model earns its place only if it improves on a simple rule. We use validation to choose it. We look at the test months only once after that choice.

**Run these cells in order.** Use the split you just created: `20260926T113753529941Z`. Here, **precision** means how many predicted late requests really were late; **recall** means how many truly late requests we found. **F1** combines both.

In [ ]:
# Cell 2 — Why: pin this run to the exact split and verify its files.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
split_id = "20260926T113753529941Z"
split_dir = root / "splits" / split_id
manifest = json.loads((split_dir / "manifest.json").read_text())
if manifest["split_id"] != split_id or set(manifest["files"]) != {"train", "validation", "test"}:
    raise ValueError("Wrong or incomplete split manifest")
for name, info in manifest["files"].items():
    raw = (split_dir / f"{name}.jsonl").read_bytes()
    if hashlib.sha256(raw).hexdigest() != info["sha256"] or len(raw.splitlines()) != info["rows"]:
        raise ValueError(f"{name} file changed or is incomplete")
print("Verified split:", split_id)
print("Rows:", {name: info["rows"] for name, info in manifest["files"].items()})

In [ ]:
# Cell 3 — Why: read training and validation; keep test answers hidden.
train = pd.read_json(split_dir / "train.jsonl", lines=True)
valid = pd.read_json(split_dir / "validation.jsonl", lines=True)
features = ["complaint_type", "borough", "month", "hour", "weekday"]
required = set(features + ["label", "unique_key"])
for name, frame in {"train": train, "validation": valid}.items():
    if not required.issubset(frame.columns) or not frame["label"].isin([0, 1]).all():
        raise ValueError(f"{name} has an unexpected schema or label")
    if frame["unique_key"].duplicated().any():
        raise ValueError(f"{name} has duplicate request IDs")
if set(train["unique_key"]) & set(valid["unique_key"]):
    raise ValueError("A request appears in both groups")
print("Inputs available when a request is created:", features)
print("Train rows:", len(train), "Validation rows:", len(valid))

In [ ]:
# Cell 4 — Why: set a minimum score using the most common training label.
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def scores(actual, predicted):
    return {
        "accuracy": round(accuracy_score(actual, predicted), 3),
        "precision_late": round(precision_score(actual, predicted, zero_division=0), 3),
        "recall_late": round(recall_score(actual, predicted, zero_division=0), 3),
        "f1_late": round(f1_score(actual, predicted, zero_division=0), 3),
    }

common_label = int(train["label"].mode().iloc[0])
baseline = scores(valid["label"], np.full(len(valid), common_label))
print("Baseline always predicts:", common_label)
print("Baseline on validation:", baseline)

In [ ]:
# Cell 5 — Why: keep missing-value handling, encoding, and model together.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

text_steps = Pipeline([("fill", SimpleImputer(strategy="constant", fill_value="MISSING")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
time_steps = Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("text", text_steps, ["complaint_type", "borough"]),
                                ("time", time_steps, ["month", "hour", "weekday"])])
candidate = Pipeline([("prepare", preprocess),
                      ("model", LogisticRegression(solver="liblinear", max_iter=1000,
                                                   random_state=42))])
print("Candidate: logistic regression with one saved preparation pipeline")

In [ ]:
# Cell 6 — Why: fit only on earlier months and choose with validation.
candidate.fit(train[features], train["label"])
valid_guess = candidate.predict(valid[features])
candidate_validation = scores(valid["label"], valid_guess)
print("Model on validation:", candidate_validation)
print("Baseline on validation:", baseline)
if candidate_validation["f1_late"] <= baseline["f1_late"]:
    raise ValueError("Model did not beat the baseline F1. Stop before opening test.")
print("Chosen for final test: logistic regression")

In [ ]:
# Cell 7 — Why: train the chosen design on all earlier months, then test once.
from sklearn.base import clone

earlier = pd.concat([train, valid], ignore_index=True)
final_model = clone(candidate).fit(earlier[features], earlier["label"])
test = pd.read_json(split_dir / "test.jsonl", lines=True)
if len(test) != manifest["files"]["test"]["rows"] or not required.issubset(test.columns):
    raise ValueError("Test file does not match its manifest")
if not test["label"].isin([0, 1]).all() or set(test["unique_key"]) & set(earlier["unique_key"]):
    raise ValueError("Test labels or request IDs are invalid")
test_scores = scores(test["label"], final_model.predict(test[features]))
print("Final model on held-out Nov–Dec test:", test_scores)

In [ ]:
# Cell 8 — Why: give the final model its own versioned folder.
from datetime import datetime, timezone
import joblib

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = root / "runs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
model_file = run_dir / "model.joblib"
joblib.dump(final_model, model_file)
model_hash = hashlib.sha256(model_file.read_bytes()).hexdigest()
print("Run ID:", run_id)
print("Saved model:", model_file)

In [ ]:
# Cell 9 — Why: record data, settings, scores, and software versions.
import platform, sklearn

run_record = {
    "run_id": run_id, "split_id": split_id,
    "source_dataset_id": manifest["source_dataset_id"],
    "features": features, "positive_label": 1, "decision_threshold": 0.5,
    "model": "LogisticRegression", "solver": "liblinear", "random_state": 42,
    "train_plus_validation_rows": len(earlier), "test_rows": len(test),
    "baseline_validation": baseline, "candidate_validation": candidate_validation,
    "final_test": test_scores, "model_sha256": model_hash,
    "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                 "numpy": np.__version__, "scikit_learn": sklearn.__version__,
                 "joblib": joblib.__version__}, "lesson_version": "stage5c-v1"
}
(run_dir / "run.json").write_text(json.dumps(run_record, indent=2) + "\n", encoding="utf-8")
print("Run ID:", run_id)
print("Model SHA-256:", run_record["model_sha256"])

# Stage 5C takeaway and quiz

**What you learned:** A majority-rule baseline gives us a minimum to beat. We trained a model using only creation-time information, chose it using September–October validation, retrained on January–October, and evaluated it once on November–December test. We saved the model and its lineage so a result can be traced back to the exact data split.

**Easy quiz:**

1. If the baseline always predicts `0`, why can its accuracy look good while its recall for late requests is zero?
2. Which months chose the model, and which months gave the final score?
3. Why do we save the preparation steps together with the model?
4. What does the `split_id` in `run.json` help us find?

Send me the **baseline validation**, **model validation**, **final test** scores, the **Run ID**, and your quiz answers. Next, Stage 6 will make training repeatable with an explicit environment and run checks.